# Nash Equilibria and Strategic Interaction
**Category: Economics**

## 1. Introduction

Game theory studies rational decision-making in settings where the outcome for each player depends on the choices of others. The central solution concept — the **Nash equilibrium** — formalises the intuition of a stable outcome: a profile of strategies from which no individual player has an incentive to deviate unilaterally.

## 2. Normal-Form Games

### 2.1 Definition

A finite normal-form game is a triple $G = (N, S, u)$ where:
- $N = \{1, \ldots, n\}$ is the set of players
- $S = S_1 \times \cdots \times S_n$, with $S_i$ the finite strategy set of player $i$
- $u = (u_1, \ldots, u_n)$, with $u_i : S \to \mathbb{R}$ the payoff function of player $i$

A **mixed strategy** for player $i$ is a probability distribution $\sigma_i \in \Delta(S_i)$. The expected payoff under a mixed profile $\sigma = (\sigma_1, \ldots, \sigma_n)$ is
$$
U_i(\sigma) = \sum_{s \in S} \left(\prod_{j=1}^n \sigma_j(s_j)\right) u_i(s)
$$

### 2.2 Nash Equilibrium

A mixed strategy profile $\sigma^*$ is a **Nash equilibrium** if for every player $i$ and every alternative strategy $\sigma_i'$,
$$
U_i(\sigma_i^*, \sigma_{-i}^*) \geq U_i(\sigma_i', \sigma_{-i}^*)
$$
where $\sigma_{-i}^*$ denotes the strategies of all players other than $i$. Nash (1950) proved that every finite game has at least one Nash equilibrium in mixed strategies.

## 3. Classic Examples

### 3.1 The Prisoner's Dilemma

Two suspects are interrogated separately. Each can **Cooperate** (stay silent) or **Defect** (testify). The payoff matrix (row player's payoff, column player's payoff) is:

|  | Cooperate | Defect |
|---|---|---|
| **Cooperate** | $(-1,\ -1)$ | $(-3,\ 0)$ |
| **Defect** | $(0,\ -3)$ | $(-2,\ -2)$ |

Defect **strictly dominates** Cooperate for each player regardless of the other's choice. The unique Nash equilibrium is (Defect, Defect) with payoff $(-2, -2)$, even though (Cooperate, Cooperate) at $(-1,-1)$ is Pareto superior. This tension between individual rationality and collective welfare is the central insight of the dilemma.

### 3.2 Matching Pennies

Two players simultaneously show a penny — Heads or Tails. Player 1 wins $\$1$ if the pennies match; Player 2 wins $\$1$ if they differ.

|  | Heads | Tails |
|---|---|---|
| **Heads** | $(1,\ -1)$ | $(-1,\ 1)$ |
| **Tails** | $(-1,\ 1)$ | $(1,\ -1)$ |

There is no pure-strategy Nash equilibrium. The unique mixed-strategy equilibrium has each player randomise $50/50$, giving expected payoff $0$ to both. This is a **zero-sum game**: $u_1(s) + u_2(s) = 0$ for all $s$.

## 4. Finding Equilibria Numerically

### 4.1 Support Enumeration for 2×2 Games

In [ ]:
import numpy as np
from itertools import product

def find_nash_2x2(A, B):
    """
    Find all Nash equilibria of a 2x2 bimatrix game.
    A: payoff matrix for row player (2x2)
    B: payoff matrix for column player (2x2)
    Returns list of (sigma1, sigma2) equilibrium pairs.
    """
    equilibria = []

    # Pure strategy equilibria
    for i, j in product(range(2), range(2)):
        row_dev = all(A[i, j] >= A[k, j] for k in range(2))
        col_dev = all(B[i, j] >= B[i, k] for k in range(2))
        if row_dev and col_dev:
            s1 = np.array([1.0 if k == i else 0.0 for k in range(2)])
            s2 = np.array([1.0 if k == j else 0.0 for k in range(2)])
            equilibria.append((s1, s2))

    # Mixed strategy equilibrium: row player mixes iff column is indifferent
    # Solve B[0,:] @ q = B[1,:] @ q for q (column's mix)
    dB = B[:, 0] - B[:, 1]
    if dB[0] != dB[1]:
        p = (B[1, 1] - B[0, 1]) / (B[0, 0] - B[1, 0] - B[0, 1] + B[1, 1]) if (B[0,0] - B[1,0] - B[0,1] + B[1,1]) != 0 else None
        q = (A[1, 1] - A[1, 0]) / (A[0, 0] - A[0, 1] - A[1, 0] + A[1, 1]) if (A[0,0] - A[0,1] - A[1,0] + A[1,1]) != 0 else None
        if p is not None and q is not None and 0 < p < 1 and 0 < q < 1:
            equilibria.append((np.array([p, 1-p]), np.array([q, 1-q])))

    return equilibria

# Prisoner's Dilemma
A_pd = np.array([[-1, -3], [0, -2]], dtype=float)
B_pd = A_pd.T
print('Prisoner\'s Dilemma equilibria:')
for s1, s2 in find_nash_2x2(A_pd, B_pd):
    print(f'  sigma1={s1}, sigma2={s2}')

# Matching Pennies
A_mp = np.array([[1, -1], [-1, 1]], dtype=float)
B_mp = -A_mp
print('Matching Pennies equilibria:')
for s1, s2 in find_nash_2x2(A_mp, B_mp):
    print(f'  sigma1={s1}, sigma2={s2}')

### 4.2 Best-Response Correspondence

In [ ]:
# Visualise best-response correspondences for Matching Pennies
q_vals = np.linspace(0, 1, 300)

# Row player's expected payoff from Heads minus Tails
# EU1(H) - EU1(T) = q*(1) + (1-q)*(-1) - [q*(-1) + (1-q)*(1)] = 4q - 2
# BR1: play H if q > 0.5, T if q < 0.5, any mix if q = 0.5
br1_p = np.where(q_vals > 0.5, 1.0, np.where(q_vals < 0.5, 0.0, 0.5))

p_vals = np.linspace(0, 1, 300)
# Column player's EU from Heads minus Tails given row plays H with prob p
# EU2(H) - EU2(T) = p*(-1) + (1-p)*(1) - [p*(1) + (1-p)*(-1)] = 2 - 4p
# BR2: play H if p < 0.5
br2_q = np.where(p_vals < 0.5, 1.0, np.where(p_vals > 0.5, 0.0, 0.5))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(br1_p, q_vals, label='BR$_1$: row player', linewidth=2)
ax.plot(p_vals, br2_q, label='BR$_2$: column player', linewidth=2, linestyle='--')
ax.plot(0.5, 0.5, 'ko', markersize=8, label='Nash equilibrium $(\\frac{1}{2}, \\frac{1}{2})$')
ax.set_xlabel('$p$ (prob. row plays Heads)')
ax.set_ylabel('$q$ (prob. col plays Heads)')
ax.set_title('Best-Response Correspondences: Matching Pennies')
ax.legend()
plt.tight_layout()
plt.show()